# Physics Informed Deep Learning (Part I): Data-driven Solutions of Nonlinear Partial Differential Equations

**Paper:** Raissi, M., Perdikaris, P., Karniadakis, G.E. (2017). *Physics Informed Deep Learning (Part I): Data-driven Solutions of Nonlinear Partial Differential Equations.* arXiv:1711.10561.

**Carpeta origen:** `PINNs/4. Otros/Physics Informed Deep Learning (Part I) Data-driven.pdf`

## Como se usan las PINNs en este paper

Este es el preprint original que precede al articulo de Raissi et al. 2019 en JCP (ya cubierto en la carpeta "1. mecanica de fluidos" de esta coleccion, con el ejemplo de la ecuacion de Burgers). Para evitar redundancia con ese cuaderno, aqui reproducimos el **segundo ejemplo del paper (Seccion 2.2): la ecuacion de Schrodinger no lineal 1D**, elegido por los autores explicitamente para mostrar que el metodo maneja **condiciones de contorno periodicas, soluciones de valor complejo, y no linealidades** distintas a las de Burgers.

$$ih_t+0.5h_{xx}+|h|^2h=0,\quad x\in[-5,5],\ t\in[0,\pi/2]$$
$$h(0,x)=2\,\text{sech}(x),\qquad h(t,-5)=h(t,5),\qquad h_x(t,-5)=h_x(t,5)$$

Como $h(t,x)=u(t,x)+iv(t,x)$ es de valor complejo, la red predice **ambas partes real e imaginaria** $[u,v]$ simultaneamente. El residuo fisico $f=f_u+if_v$ (Eq. 2, especializado a esta EDP) se separa en dos residuos reales:

$$f_u := -u_t-0.5v_{xx}-(u^2+v^2)v,\qquad f_v := -v_t+0.5u_{xx}+(u^2+v^2)u$$

y la perdida (Eq. 4) combina $MSE_0$ (condicion inicial), $MSE_b$ (frontera periodica en $h$ y $h_x$) y $MSE_f$ (residuo de $f_u,f_v$ en los puntos de colocacion), igual que en el ejemplo de Burgers pero adaptada a valores complejos y periodicidad.

Este cuaderno reproduce fielmente la red (arquitectura similar: MLP profunda, tanh, 2 salidas $[u,v]$), la funcion de perdida completa (IC + BC periodica + residuo PDE), y genera una solucion de referencia de alta precision mediante un **metodo espectral de paso dividido (split-step Fourier)** &mdash; el metodo numerico estandar para la ecuacion de Schrodinger no lineal &mdash; para validar la prediccion de la PINN.

## Repositorio publico

El paper **incluye explicitamente** el enlace a su repositorio oficial en el propio texto (Seccion 2): "All code and data-sets accompanying this manuscript are available at https://github.com/maziarraissi/PINNs".

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs (carpeta `appendix/continuous_time_inference (Schrodinger)/`)

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Solucion de referencia: metodo espectral de paso dividido (split-step Fourier) para la NLS

In [ ]:
L, N_x = 10.0, 512
x_grid = np.linspace(-5, 5, N_x, endpoint=False)
dx = x_grid[1] - x_grid[0]
k_wave = 2 * np.pi * np.fft.fftfreq(N_x, d=dx)

T_final = np.pi / 2
n_steps = 4000
dt = T_final / n_steps

h = (2.0 / np.cosh(x_grid)).astype(np.complex128)  # h(0,x) = 2 sech(x)
snapshots = {0.0: h.copy()}
save_times = np.linspace(0, T_final, 100)
save_idx = 0
h_history = [h.copy()]
t_history = [0.0]

linear_half = np.exp(-1j * 0.5 * k_wave**2 * (dt / 2))  # paso lineal (medio), en espacio de Fourier

for step in range(1, n_steps + 1):
    h_hat = np.fft.fft(h)
    h_hat *= linear_half
    h = np.fft.ifft(h_hat)
    h *= np.exp(1j * np.abs(h)**2 * dt)                  # paso no lineal (exacto)
    h_hat = np.fft.fft(h)
    h_hat *= linear_half
    h = np.fft.ifft(h_hat)
    if step % (n_steps // 99) == 0:
        h_history.append(h.copy())
        t_history.append(step * dt)

h_history = np.array(h_history)   # (n_snap, N_x)
t_history = np.array(t_history)
print(f'Referencia generada: {h_history.shape[0]} instantes, {N_x} puntos espaciales')

## 2. Red PINN: dos salidas $[u,v]$ (partes real e imaginaria de $h$)

In [ ]:
class SchrodingerPINN(nn.Module):
    def __init__(self, n_hidden=6, n_neurons=50):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 2)]
        self.net = nn.Sequential(*layers)

    def forward(self, t, x):
        out = self.net(torch.cat([t, x], dim=1))
        return out[:, 0:1], out[:, 1:2]  # u, v


model = SchrodingerPINN().to(device)


def d_d(f, v):
    return torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Perdida (Eq. 4): $MSE_0$ (condicion inicial) + $MSE_b$ (frontera periodica en $h$ y $h_x$) + $MSE_f$ (residuo PDE)

In [ ]:
N_0, N_b, N_f = 50, 50, 4000

x0_np = np.random.uniform(-5, 5, N_0)
u0_t = torch.tensor(2 / np.cosh(x0_np), dtype=torch.float32, device=device).view(-1, 1)
v0_t = torch.zeros_like(u0_t)
x0_t = torch.tensor(x0_np, dtype=torch.float32, device=device).view(-1, 1)
t0_t = torch.zeros_like(x0_t)

t_b_np = np.random.uniform(0, T_final, N_b)
t_b_t = torch.tensor(t_b_np, dtype=torch.float32, device=device).view(-1, 1)
xL_t = torch.full((N_b, 1), -5.0, device=device, requires_grad=True)
xR_t = torch.full((N_b, 1), 5.0, device=device, requires_grad=True)

t_f = (torch.rand(N_f, 1, device=device) * T_final).requires_grad_(True)
x_f = (torch.rand(N_f, 1, device=device) * 10 - 5).requires_grad_(True)


def compute_loss(model):
    u0_p, v0_p = model(t0_t, x0_t)
    loss_0 = torch.mean((u0_p - u0_t)**2 + (v0_p - v0_t)**2)

    uL, vL = model(t_b_t, xL_t)
    uR, vR = model(t_b_t, xR_t)
    uL_x = d_d(uL, xL_t); vL_x = d_d(vL, xL_t)
    uR_x = d_d(uR, xR_t); vR_x = d_d(vR, xR_t)
    loss_b = torch.mean((uL - uR)**2 + (vL - vR)**2 + (uL_x - uR_x)**2 + (vL_x - vR_x)**2)

    u, v = model(t_f, x_f)
    u_t = d_d(u, t_f); v_t = d_d(v, t_f)
    u_x = d_d(u, x_f); v_x = d_d(v, x_f)
    u_xx = d_d(u_x, x_f); v_xx = d_d(v_x, x_f)
    f_u = -u_t - 0.5 * v_xx - (u**2 + v**2) * v
    f_v = -v_t + 0.5 * u_xx + (u**2 + v**2) * u
    loss_f = torch.mean(f_u**2 + f_v**2)

    return loss_0 + loss_b + loss_f, loss_0.item(), loss_b.item(), loss_f.item()

## 4. Entrenamiento

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(6000):
    optimizer.zero_grad()
    loss, l0, lb, lf = compute_loss(model)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 1000 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | L0={l0:.4e} | Lb={lb:.4e} | Lf={lf:.4e}')

## 5. Resultados: $|h(t,x)|$ predicho vs. referencia espectral (cf. Fig. de la Seccion 2.2 del paper)

In [ ]:
Xg, Tg = np.meshgrid(x_grid, t_history)
tx_t = torch.tensor(Tg.ravel(), dtype=torch.float32, device=device).view(-1, 1)
tx_x = torch.tensor(Xg.ravel(), dtype=torch.float32, device=device).view(-1, 1)
with torch.no_grad():
    u_pred, v_pred = model(tx_t, tx_x)
h_pred_mag = torch.sqrt(u_pred**2 + v_pred**2).cpu().numpy().reshape(Xg.shape)
h_ref_mag = np.abs(h_history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].pcolormesh(Tg, Xg, h_ref_mag, cmap='viridis', shading='auto')
axes[0].set_title('|h(t,x)| referencia (split-step)'); axes[0].set_xlabel('t'); axes[0].set_ylabel('x')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(Tg, Xg, h_pred_mag, cmap='viridis', shading='auto')
axes[1].set_title('|h(t,x)| PINN'); axes[1].set_xlabel('t')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].pcolormesh(Tg, Xg, np.abs(h_pred_mag - h_ref_mag), cmap='inferno', shading='auto')
axes[2].set_title('|error|'); axes[2].set_xlabel('t')
plt.colorbar(im2, ax=axes[2])
plt.tight_layout()
plt.show()

err = 100 * np.linalg.norm(h_pred_mag - h_ref_mag) / np.linalg.norm(h_ref_mag)
print(f'Error relativo L2 en |h(t,x)|: {err:.2f}%  (paper reporta 1.97e-3 en norma L2 compleja completa)')